In [1]:
import os
os.chdir('/workspace/ea35b3fb-1b6a-4c62-bcf8-209857918dcc')
print(os.listdir('.'))


['.prompts', 'lchi5_zeros_N5000_dps80.npy', '.kernel_llm_logs_1.txt', '-PROMPT-v6-DATASET.md', '.config', 'zeta_zeros_N5000_dps50.npy', 'ldh_zeros_N5000_dps50.npy', 'memory', 'ldh_off_line_zeros.csv']


In [2]:
import numpy as np
zeta = np.load('zeta_zeros_N5000_dps50.npy')
ldh = np.load('ldh_zeros_N5000_dps50.npy')
lchi5 = np.load('lchi5_zeros_N5000_dps80.npy')
print(zeta.shape, zeta.dtype, zeta[0][:30])
print(ldh.shape, ldh.dtype, ldh[0][:30])
print(lchi5.shape, lchi5.dtype, lchi5[0][:30])


(5000,) <U51 14.134725141734693790457251983
(5000,) <U51 5.0941598445710949256987955170
(5000,) <U81 6.6484533447277147161232784599


In [3]:

# Plan:
# We have 3 complete N=5000 zero lists: zeta (dps50), L(chi5) (dps80), L_DH (dps50).
# L(Delta) and zeta_delta are NOT available locally (per dataset description, L(Delta) is incomplete,
# zeta_delta would need construction). Per primary-dataset notes, zeta_delta_strong_N5000_dps50.npy
# was mentioned but I don't see it on disk. Let me check.
import os
for f in os.listdir('.'):
 print(f, os.path.getsize(f) if os.path.isfile(f) else 'dir')


.prompts dir
lchi5_zeros_N5000_dps80.npy 1620128
.kernel_llm_logs_1.txt 0
-PROMPT-v6-DATASET.md 25848
.config dir
zeta_zeros_N5000_dps50.npy 1020128
ldh_zeros_N5000_dps50.npy 1020128
memory dir
ldh_off_line_zeros.csv 18101


In [4]:

# We have 3 of the 5 zero lists. We need L(Delta) zeros and we should construct zeta_delta.
# Let me try to fetch L(Delta) zeros from LMFDB (1.12.a.a) - this is the Ramanujan delta L-function.
# I'll attempt to retrieve at least the first few thousand zeros.
# Then construct zeta_delta_strong from zeta zeros via known perturbation pattern.

# Actually, the primary task: compute moments I_k(T) and compare to Keating-Snaith.
# Let's first analyze what we can do with available data:
# - zeta (5000 zeros, dps50) - U(N) / unitary symmetry
# - L(chi4 mod 5) (5000 zeros, dps80) - U(N) / unitary
# - L_DH (5000 zeros, dps50) - has off-line zeros, no Euler product
# - L(Delta) - NOT available; would need to retrieve from LMFDB
# - zeta_delta - NOT directly available

# For L(Delta), I'll attempt LMFDB retrieval. For zeta_delta, we can construct ourselves.

# Let me think about the moment computation method.
# I_k(T) = (1/T) ∫_0^T |L(1/2+it)|^{2k} dt for k=1,2 
# (Note: research-objective says |L|^k for k=1,2, but standard RMT convention is 2k-th moment.
# Task #7 in dataset says I_k(T)=∫₀ᵀ|ζ(1/2+it)|^{2k}dt for k=1,2.)
# 
# We have ZEROS, not L-values. Computing the integral of |L|^{2k} from zeros via Guinand-Weil
# is nontrivial. The objective says "use Guinand-Weil to compute moments via sums over zeros".
#
# Actually the right approach: compute |L(1/2+it)| at many sample points using the Hadamard-product-
# style formula from zeros (truncated). This is essentially Riemann-Siegel/Hardy Z evaluation, but
# we can use mpmath siegelz / lfunhardy for ground truth, OR approximate from zeros.
#
# Practical approach: sample |L(1/2+it)| at fine grid using mpmath functions directly
# (zeta for zeta; we have arithmetic for L(chi5) via mpmath dirichlet, and lfun for L_DH via 
# cypari2). Then numerically integrate to get I_k(T).
#
# Given compute constraints, let's use mpmath.zeta for zeta. For L(chi5), use mpmath.dirichlet.
# For L_DH, define it explicitly.
# For L(Delta), we'd need lfunhardy via cypari2 (expensive).
#
# This research-objective is large. Let me focus on the 3 available functions and report 
# what cannot be done for L(Delta) and zeta_delta (per the dataset description, L(Delta) at 
# N=5000 is incomplete due to extreme cost, and zeta_delta_strong is on disk per docs but 
# missing here).
print("Plan formed")


Plan formed


In [5]:

# Verify R2 gate first: γ1=14.134725141, γ2=21.022039639, γ3=25.010857580
import mpmath as mp
mp.mp.dps = 30
for i in range(3):
 print(f"zeta zero {i+1}: {zeta[i][:25]}")
print()
print("Reference: 14.134725141, 21.022039639, 25.010857580")


zeta zero 1: 14.1347251417346937904572
zeta zero 2: 21.0220396387715549926284
zeta zero 3: 25.0108575801456887632137

Reference: 14.134725141, 21.022039639, 25.010857580


In [6]:

# Compute moments via direct integration of |L(1/2+it)| using mpmath functions.
# We'll use a fine grid. For the heights, we'll go up to T = max imaginary part of N=5000-th zero.
# Then compute I_k(T) at multiple T values and compare to RMT predictions.

# Approach (avoiding huge mpmath cost):
# Use numpy float64 approximations via mpmath for sampling — but mpmath is slow.
# Better: use scipy.special or compute with mpmath at reasonable precision (15 digits is fine 
# for moment estimation).

# Strategy: use mpmath.zeta with workdps=20, sample at e.g. dt=0.05.
# T_max ≈ γ_5000 ≈ ?
print("Last zeta zero ~", float(zeta[-1].split('.')[0]+'.'+zeta[-1].split('.')[1][:10]))
print("Last L_DH zero ~", float(ldh[-1].split('.')[0]+'.'+ldh[-1].split('.')[1][:10]))
print("Last L(chi5) zero ~", float(lchi5[-1].split('.')[0]+'.'+lchi5[-1].split('.')[1][:10]))


Last zeta zero ~ 5447.8619983012
Last L_DH zero ~ 4981.1674898625
Last L(chi5) zero ~ 4388.2090125941


In [7]:

# T_max ≈ 4400 (smallest among the three). Number of samples at dt=0.05 → 88,000 points × 3 funcs.
# mpmath.zeta calls at 88k points: maybe slow but feasible.
# Use dt=0.1 → 44,000 points to be safer.

# Let me time mpmath.zeta and decide.
import time, mpmath as mp
mp.mp.dps = 20
t0 = time.time()
n_test = 200
for k in range(n_test):
 mp.zeta(mp.mpc(0.5, 100+k*0.1))
print("zeta:", (time.time()-t0)/n_test*1000, "ms each")


zeta: 3.082270622253418 ms each


In [8]:

# 3ms × 44000 = 132s per function. Manageable.
# But at large t (~4000), mpmath.zeta uses Riemann-Siegel and is much faster actually.
# Let's also use scipy's higher-throughput options. Actually, mpmath at low dps is what we have.

# For L(chi5 mod 5), use mpmath.dirichlet
import mpmath as mp
mp.mp.dps = 20
# chi4 mod 5: chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1, chi(0)=0 (order-4 character)
# mpmath.dirichlet(s, chi) where chi is list of length q

# Let's test:
chi5_vals = [0, 1, 1j, -1j, -1] # values at 0,1,2,3,4 mod 5
val = mp.dirichlet(mp.mpc(0.5, 14), chi5_vals)
print("L(chi5, 0.5+14i) =", val, "|.|=", abs(val))

t0 = time.time()
for k in range(100):
 mp.dirichlet(mp.mpc(0.5, 100+k*0.1), chi5_vals)
print("L(chi5):", (time.time()-t0)/100*1000, "ms each")


L(chi5, 0.5+14i) = (1.090064759586415505 - 0.69401530470597268719j) |.|= 1.2922454965130712145


L(chi5): 16.54815912246704 ms each


In [9]:

# L(chi5) is slower. 16ms × 44000 = ~12 minutes. Still OK.

# For L_DH (Davenport-Heilbronn): no Euler product. Standard definition:
# L_DH(s) = 5^(-s) [ ξ ζ(s, 1/5) + ξ̄ ζ(s, 2/5) - ξ̄ ζ(s, 3/5) - ξ ζ(s, 4/5) ]
# where ξ is a specific constant... Actually the more standard form:
# L_DH(s) = ζ(s,1/5) + κ ζ(s,2/5) - κ ζ(s,3/5) - ζ(s,4/5)
# normalized so that κ = (sqrt(10-2√5)-2)/(√5-1)
# This satisfies functional equation but no Euler product.

# Compute κ:
mp.mp.dps = 30
kappa = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2) / (mp.sqrt(5) - 1)
print("kappa =", kappa)

# L_DH at known zero (0.808517, 85.699348) should be ~0
s_test = mp.mpc("0.808517", "85.699348")
def L_DH(s, q=5):
 return mp.zeta(s, mp.mpf(1)/5) + kappa*mp.zeta(s, mp.mpf(2)/5) - kappa*mp.zeta(s, mp.mpf(3)/5) - mp.zeta(s, mp.mpf(4)/5)

# Note: there's also a 5^{-s} prefactor in some definitions. Let's check with the documented off-line zero.
val = L_DH(s_test)
print("L_DH at off-line zero:", val, "|.|=", abs(val))

# Also check at a critical line zero from our list:
s_test2 = mp.mpc("0.5", ldh[0])
val2 = L_DH(s_test2)
print("L_DH at first critical-line zero:", val2, "|.|=", abs(val2))


kappa = 0.284079043840412296028291832393
L_DH at off-line zero: (-0.00000180581518009257566134077216861 - 0.00000156910728900960884480312219763j) |.|= 0.00000239229307340799396608387708965
L_DH at first critical-line zero: (2.46519032881566189191165176651e-32 - 3.94430452610505902705864282641e-31j) |.|= 3.95200073737666665681875745999e-31


In [10]:

# Excellent! L_DH definition is correct. R2 gate passes (|L_DH(off-line zero)| < 1e-5 ≈ 2.4e-6).
# Critical-line zero gives essentially 0 (3e-31). Perfect.

# Now let's time L_DH:
t0 = time.time()
for k in range(50):
 L_DH(mp.mpc(0.5, 100+k*0.1))
print("L_DH:", (time.time()-t0)/50*1000, "ms each")


L_DH: 19.554057121276855 ms each


In [11]:

# ~20ms × 44000 = ~15 min for L_DH at full T range. Total ~30 min for L(chi5)+L_DH.
# Let's reduce: use dt=0.2 → 22,000 points per func, ~7 min each.

# Actually, since we want I_k(T) at several T to make the log-log plot, we just need to compute 
# |L|^{2k} once on the full grid, then take cumulative averages at increasing T.

# Set up:
# T_max = 4400 (~ smaller than smallest last zero across all funcs)
# Sample on uniform grid [0, T_max]
# Compute integrand on grid, then cum-sum to get I_k(T) for grid of T values

# Choose dt = 0.2. ~22000 points per function.

import numpy as np
import mpmath as mp
import time

T_max = 4000.0 # below smallest last γ
dt = 0.2
t_grid = np.arange(dt, T_max+dt/2, dt) # avoid t=0 (pole)
N_pts = len(t_grid)
print(f"N_pts = {N_pts}, T_max = {t_grid[-1]}")


N_pts = 20000, T_max = 4000.0


In [12]:

# Compute |zeta(1/2+it)| at each grid point
mp.mp.dps = 20

# Use mpmath.siegelz which is fast and gives |Z(t)| = |zeta(1/2+it)|
t0 = time.time()
abs_zeta = np.empty(N_pts)
for i, t in enumerate(t_grid):
 abs_zeta[i] = abs(float(mp.siegelz(t)))
 if i % 5000 == 0:
 print(f" zeta {i}/{N_pts} elapsed {time.time()-t0:.1f}s")
print(f"zeta done in {time.time()-t0:.1f}s, mean |zeta| = {abs_zeta.mean():.3f}")


 zeta 0/20000 elapsed 0.0s


 zeta 5000/20000 elapsed 36.0s


 zeta 10000/20000 elapsed 99.5s


 zeta 15000/20000 elapsed 177.7s


zeta done in 273.8s, mean |zeta| = 1.777


In [13]:

# 4.5 min for zeta. L(chi5) and L_DH likely 10-15 min each. Let's parallelize via multiprocessing.
# Actually mpmath isn't multiprocessing-friendly but should work. Let me try.

# Faster idea: lower dps further? mp.mp.dps=15. Already at 20.
# Or use a coarser grid. Actually 273s is fine. Let me just do them sequentially.

# Save zeta values.
np.save('abs_zeta_grid.npy', abs_zeta)

# Now L(chi5 mod 5):
t0 = time.time()
abs_lchi5 = np.empty(N_pts)
chi5_vals = [0, 1, 1j, -1j, -1]
for i, t in enumerate(t_grid):
 abs_lchi5[i] = abs(complex(mp.dirichlet(mp.mpc(0.5, t), chi5_vals)))
 if i % 2500 == 0:
 print(f" lchi5 {i}/{N_pts} elapsed {time.time()-t0:.1f}s")
print(f"lchi5 done in {time.time()-t0:.1f}s, mean = {abs_lchi5.mean():.3f}")
np.save('abs_lchi5_grid.npy', abs_lchi5)


TimeoutError: Code execution timed out after 900.0 seconds

In [14]:

# Took too long. Let me check progress.
import os
print(os.path.exists('abs_lchi5_grid.npy'))
# Probably didn't complete. Use parallel processing.


False


In [15]:

# Let me parallelize using ProcessPoolExecutor.
import os
n_cpu = os.cpu_count()
print("CPUs:", n_cpu)


CPUs: 8


In [16]:

# Write helper module and use multiprocessing
helper_code = '''
import mpmath as mp
import numpy as np

mp.mp.dps = 20
chi5_vals = [0, 1, 1j, -1j, -1]
kappa = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2) / (mp.sqrt(5) - 1)

def abs_zeta_at(t):
 return abs(float(mp.siegelz(t)))

def abs_lchi5_at(t):
 return abs(complex(mp.dirichlet(mp.mpc(0.5, t), chi5_vals)))

def abs_ldh_at(t):
 s = mp.mpc(0.5, t)
 v = mp.zeta(s, mp.mpf(1)/5) + kappa*mp.zeta(s, mp.mpf(2)/5) - kappa*mp.zeta(s, mp.mpf(3)/5) - mp.zeta(s, mp.mpf(4)/5)
 return abs(complex(v))

def batch_zeta(ts):
 return [abs_zeta_at(t) for t in ts]

def batch_lchi5(ts):
 return [abs_lchi5_at(t) for t in ts]

def batch_ldh(ts):
 return [abs_ldh_at(t) for t in ts]
'''
with open('lf_helpers.py', 'w') as f:
 f.write(helper_code)
print("Helper module written")


Helper module written


In [17]:

# Use multiprocessing
import multiprocessing as mpc
import numpy as np
import time
import sys
sys.path.insert(0, '.')

t_grid = np.arange(0.2, 4000+0.1, 0.2)
N_pts = len(t_grid)
print("N_pts:", N_pts)

# Chunk by interleaving so workers have balanced cost (cost grows with t, so block partitioning is bad)
n_workers = 8
# Cost grows roughly linearly with t for these functions. Use interleaving.
chunks = [t_grid[i::n_workers] for i in range(n_workers)]
print("chunk sizes:", [len(c) for c in chunks])


N_pts: 20000
chunk sizes: [2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500]


In [18]:

# Compute L(chi5) in parallel
from multiprocessing import Pool
import lf_helpers
import importlib
importlib.reload(lf_helpers)

t0 = time.time()
with Pool(n_workers) as pool:
 results = pool.map(lf_helpers.batch_lchi5, [list(c) for c in chunks])
elapsed = time.time() - t0
print(f"lchi5 parallel done in {elapsed:.1f}s")

# Reassemble
abs_lchi5 = np.empty(N_pts)
for i, res in enumerate(results):
 abs_lchi5[i::n_workers] = res
print(f"mean |L(chi5)| = {abs_lchi5.mean():.3f}")
np.save('abs_lchi5_grid.npy', abs_lchi5)


lchi5 parallel done in 612.0s
mean |L(chi5)| = 1.810


In [19]:

# 10 min. Now L_DH similarly.
t0 = time.time()
with Pool(n_workers) as pool:
 results = pool.map(lf_helpers.batch_ldh, [list(c) for c in chunks])
elapsed = time.time() - t0
print(f"ldh parallel done in {elapsed:.1f}s")

abs_ldh = np.empty(N_pts)
for i, res in enumerate(results):
 abs_ldh[i::n_workers] = res
print(f"mean |L_DH| = {abs_ldh.mean():.3f}")
np.save('abs_ldh_grid.npy', abs_ldh)


ldh parallel done in 611.0s
mean |L_DH| = 3.609


In [20]:

import numpy as np
abs_zeta = np.load('abs_zeta_grid.npy')
abs_lchi5 = np.load('abs_lchi5_grid.npy')
abs_ldh = np.load('abs_ldh_grid.npy')
t_grid = np.arange(0.2, 4000+0.1, 0.2)
dt = 0.2
print(len(abs_zeta), len(abs_lchi5), len(abs_ldh), len(t_grid))

# Compute moments I_k(T) = (1/T) ∫_0^T |L|^{2k} dt for k=1,2
# at multiple T values.

# Use cumulative trapezoidal integration
def cum_moment(abs_vals, k):
 integrand = abs_vals ** (2*k)
 # cumulative trapezoid using dt
 cum = np.concatenate(([0.0], np.cumsum(0.5*(integrand[:-1] + integrand[1:]) * dt)))
 # cum has length N_pts; cum[i] = ∫_{t_grid[0]}^{t_grid[i]}
 # T values: t_grid[1:] (shifted)
 T_vals = t_grid # approximate T as t at each point
 # I_k(T) = cum[i] / T_vals[i] (skip i=0)
 I = np.zeros_like(cum)
 I[1:] = cum[1:] / T_vals[1:]
 return T_vals, I

T_z, I1_z = cum_moment(abs_zeta, 1)
_, I2_z = cum_moment(abs_zeta, 2)
T_c, I1_c = cum_moment(abs_lchi5, 1)
_, I2_c = cum_moment(abs_lchi5, 2)
T_d, I1_d = cum_moment(abs_ldh, 1)
_, I2_d = cum_moment(abs_ldh, 2)

print(f"At T={T_z[-1]:.0f}:")
print(f" zeta: I1={I1_z[-1]:.4f} I2={I2_z[-1]:.4f}")
print(f" lchi5: I1={I1_c[-1]:.4f} I2={I2_c[-1]:.4f}")
print(f" ldh: I1={I1_d[-1]:.4f} I2={I2_d[-1]:.4f}")


20000 20000 20000 20000
At T=4000:
 zeta: I1=6.6285 I2=272.3637
 lchi5: I1=7.2222 I2=379.9919
 ldh: I1=21.4883 I2=1707.6821


In [21]:

# Wait - the moment hypothesis in the research-objective says I_k = (1/T) ∫ |L|^k dt for k=1,2.
# But the standard Keating-Snaith formulation uses 2k-th moment.
# Let me compute BOTH I_k for k=1,2 AND M_k = (1/T)∫|L|^{2k}dt for clarity.

# I'll report using the research-objective's k=1,2 of |L|^k:
def cum_moment_k(abs_vals, k):
 """I_k(T) = (1/T) ∫_0^T |L|^k dt"""
 integrand = abs_vals ** k
 cum = np.concatenate(([0.0], np.cumsum(0.5*(integrand[:-1] + integrand[1:]) * dt)))
 I = np.zeros_like(cum)
 I[1:] = cum[1:] / t_grid[1:]
 return I

I1_z = cum_moment_k(abs_zeta, 1)
I2_z = cum_moment_k(abs_zeta, 2)
I1_c = cum_moment_k(abs_lchi5, 1)
I2_c = cum_moment_k(abs_lchi5, 2)
I1_d = cum_moment_k(abs_ldh, 1)
I2_d = cum_moment_k(abs_ldh, 2)

# Also 4th moment (which is 2k=4, k=2 in standard convention) for comparison
M4_z = cum_moment_k(abs_zeta, 4)
M4_c = cum_moment_k(abs_lchi5, 4)
M4_d = cum_moment_k(abs_ldh, 4)

print("At T=4000:")
print(f" ζ : I1(|L|^1)={I1_z[-1]:.4f} I2(|L|^2)={I2_z[-1]:.4f} M4(|L|^4)={M4_z[-1]:.4f}")
print(f" L(χ5) : I1={I1_c[-1]:.4f} I2={I2_c[-1]:.4f} M4={M4_c[-1]:.4f}")
print(f" L_DH : I1={I1_d[-1]:.4f} I2={I2_d[-1]:.4f} M4={M4_d[-1]:.4f}")


At T=4000:
 ζ : I1(|L|^1)=1.7765 I2(|L|^2)=6.6285 M4(|L|^4)=272.3637
 L(χ5) : I1=1.8095 I2=7.2222 M4=379.9919
 L_DH : I1=3.6089 I2=21.4883 M4=1707.6821


In [22]:

# Keating-Snaith for unitary symmetry U(N):
# Standard result: (1/T) ∫_0^T |ζ(1/2+it)|^{2k} dt ~ a_k * g_k * (log T)^{k^2}
# where g_k = G(k+1)^2 / G(2k+1) (Keating-Snaith constant)
# and a_k is arithmetic factor (Conrey-Ghosh).
#
# For k=1: a_1=1, g_1=1, so I should be ~ log T
# For k=2: a_2 = (6/π²) * Σ ... known a_2(ζ)=6/π² approximately... actually:
# a_2(ζ) = (6/π²) * ∏_p ... wait, let me recall:
# 
# Conjecturally: (1/T)∫|ζ|^{2k}dt ~ c_k (log T)^{k^2}
# c_1 = 1
# c_2 ≈ 0.2226... (specifically a_2*g_2 where a_2=6/π² and g_2=1/12... no)
# Let me compute Keating-Snaith g_k:
# G(z) is Barnes G-function. g_k = G(k+1)^2/G(2k+1).
# For integer k: g_k = ∏_{j=0}^{k-1} j! / (k+j)! -- no, let me just compute.

import mpmath as mp
mp.mp.dps = 30
def barnesG(z):
 return mp.barnesg(z)

def gk(k):
 return barnesG(k+1)**2 / barnesG(2*k+1)

print("g_1 =", gk(1)) # should be 1
print("g_2 =", gk(2)) # should be 1/12
print("g_3 =", gk(3)) # should be 42/9! = ?

# Arithmetic factors a_k for ζ(s) (Conrey-Ghosh):
# a_k = ∏_p (1-1/p)^{k^2} * Σ_{m≥0} d_k(p^m)^2 / p^m
# where d_k is k-th divisor function.
# For k=1: a_1 = 1
# For k=2: a_2 = ∏_p (1-1/p)^4 * (1 + 4/p + 9/p^2 + 16/p^3 + ...) = ∏_p (1-1/p)^3 (1+1/p)
# Wait that's the 4th moment. Actually a_2(ζ) = 6/π²? Let me look up.
# Standard: 4th moment constant for ζ is (1/2π²) * (log T)^4 ... no
# The 4th moment of zeta: (1/T)∫|ζ|^4 ~ (1/(2π²)) (log T)^4
# So total constant = 1/(2π²) ≈ 0.0507
# And gk(2)*ak(2): gk(2) = 1/12, so ak(2) = 12/(2π²) = 6/π² ≈ 0.6079

# 2nd moment of zeta (k=1 case): (1/T)∫|ζ|^2 ~ log T, with constant 1.
print("\nKeating-Snaith RMT prediction (k=1 → |ζ|^2 has constant 1*1 = 1)")
print("Keating-Snaith RMT prediction (k=2 → |ζ|^4 has constant 6/π² * 1/12 = 1/(2π²)) =", float(1/(2*mp.pi**2)))


g_1 = 1.0
g_2 = 0.0833333333333333333333333333333
g_3 = 0.000115740740740740740740740740741

Keating-Snaith RMT prediction (k=1 → |ζ|^2 has constant 1*1 = 1)
Keating-Snaith RMT prediction (k=2 → |ζ|^4 has constant 6/π² * 1/12 = 1/(2π²)) = 0.05066059182116889


In [23]:

# So in the standard formulation:
# M_{2k}(T) := (1/T)∫|L|^{2k} dt ~ c_k * (log T)^{k^2}
# For ζ: c_1 = 1, c_2 = 1/(2π²) ≈ 0.05066
# For L(χ): same form. For primitive Dirichlet L-function, c_1 = 1 (still U(N) symmetry, 
# M_2 ~ (1/φ(q)) * Σ ... ; actually for L(χ4 mod 5) which is a single primitive character,
# the 2nd moment is (1/T)∫|L(χ,1/2+it)|^2 dt ~ (φ(q)/q) * log T = (4/5) log T.
# Actually no: for a SINGLE primitive Dirichlet L-function, M_2 ~ log T (with a different constant).
# The classical result: ∫₀^T |L(χ,1/2+it)|² dt ~ (φ(q)/q) T log T (for primitive χ mod q).
# So c_1(L(χ5)) = φ(5)/5 = 4/5 = 0.8.

# For L_DH: no Euler product, no RMT prediction directly. But its mean square is still computable
# via approximate functional equation; expected to grow but with different constant.

# Now research-objective uses I_k for k=1,2 with |L|^k (not |L|^{2k}). So:
# I_1 = (1/T)∫|L| dt and I_2 = (1/T)∫|L|^2 dt.
# k=2 in research-objective = standard 2nd moment.
# k=1 = first absolute moment, no clean Keating-Snaith form.
# 
# Per the dataset markdown (item 7), the prescribed I_k(T) is the 2k-th moment.
# So I'll interpret the research-objective using BOTH conventions, prioritizing the 
# established 2k-th moment that has RMT predictions.

# Standard moments:
# M2(T) = (1/T)∫|L|^2 for k=1 → RMT: ~ c_1 (log T)^1
# M4(T) = (1/T)∫|L|^4 for k=2 → RMT: ~ c_2 (log T)^4

# Compute predicted leading constants per function:
import numpy as np
import mpmath as mp

# Constants:
# ζ: c_1=1, c_2 = 1/(2π²)
# L(χ4 mod 5): 2nd moment leading constant = φ(5)/5 = 4/5; 
# 4th moment constant via Heath-Brown: c_2 = (1/(2π²)) * (φ(q)/q)^? -- actually for the
# k=2 KS-prediction of degree-1 L-functions, the constant is 
# a_k(L(χ)) * g_k where a_k(L(χ)) for primitive char:
# a_2(χ) = (1/(2π²)) * (φ(q)/q)^something
# This is getting complicated. Let me use empirical ratios:

# Compute ratio R_k(T) := M_{2k}(T) / (log T)^{k^2}
# This should approach c_k.

import numpy as np
logT = np.log(np.where(t_grid > 1, t_grid, 1))

# 2nd moment ratios (k=1, divisor (log T)^1)
R1_z = I2_z / np.where(logT>0.01, logT, 1) # I2 is (1/T)∫|L|^2 = "M_2"
R1_c = I2_c / np.where(logT>0.01, logT, 1)
R1_d = I2_d / np.where(logT>0.01, logT, 1)

# 4th moment ratios (k=2, divisor (log T)^4)
R2_z = M4_z / np.where(logT>0.01, logT**4, 1)
R2_c = M4_c / np.where(logT>0.01, logT**4, 1)
R2_d = M4_d / np.where(logT>0.01, logT**4, 1)

# Read at T=4000
idx = -1
print(f"At T={t_grid[idx]:.0f}, log T = {logT[idx]:.4f}")
print(f" ζ: M2/log T = {R1_z[idx]:.4f} (RMT: 1.0)")
print(f" L(χ5): M2/log T = {R1_c[idx]:.4f} (RMT: 0.8 = φ(5)/5)")
print(f" L_DH: M2/log T = {R1_d[idx]:.4f} (no RMT)")
print()
print(f" ζ: M4/(log T)^4 = {R2_z[idx]:.5f} (RMT: 1/(2π²) ≈ 0.05066)")
print(f" L(χ5): M4/(log T)^4 = {R2_c[idx]:.5f} (RMT analog ~ a_2(χ)/12)")
print(f" L_DH: M4/(log T)^4 = {R2_d[idx]:.5f} (no RMT)")


At T=4000, log T = 8.2940
 ζ: M2/log T = 0.7992 (RMT: 1.0)
 L(χ5): M2/log T = 0.8708 (RMT: 0.8 = φ(5)/5)
 L_DH: M2/log T = 2.5908 (no RMT)

 ζ: M4/(log T)^4 = 0.05755 (RMT: 1/(2π²) ≈ 0.05066)
 L(χ5): M4/(log T)^4 = 0.08030 (RMT analog ~ a_2(χ)/12)
 L_DH: M4/(log T)^4 = 0.36086 (no RMT)


In [24]:

# Results are sensible:
# ζ: M2/log T = 0.799 vs RMT 1.0 (slow convergence: actual is log T + 2γ-1-log(2π) ≈ log T - 1.8, 
# so at T=4000 with logT=8.3, predicted M2 ≈ 6.5; actual 6.628 → ratio 0.799 ✓ consistent)
# L(χ5): M2/log T = 0.871 vs analog 0.8 (very close)
# L_DH: M2/log T = 2.59, much higher → DEVIATION
#
# 4th moment for ζ: 0.0576 vs 0.0507 (within ~14% - finite size effect; convergence is slow ~1/log T)
# 4th moment for L_DH: 0.361 - very large, severely deviates
#
# For L(χ5), the predicted 4th moment leading constant for a single primitive Dirichlet L-function 
# is (1/(2π²)) * a_2(χ) where a_2(χ) is an Euler product factor.
# Actually for primitive χ mod q, the 4th moment constant is 
# c_2(χ) = (1/(2π²)) * ∏_p (some factor). For q=5, this might give ~0.08.
# Let's also confirm using known asymptotics: for L(χ,s) with primitive χ mod q,
# ∫₀^T |L|^4 ~ c_2(χ) T (log T)^4
# where c_2(χ) = (φ(q)^2 / q^2) * (1/(2π²)) * (Euler product factor)
# = (16/25)*0.0507 ≈ 0.0324 — that doesn't match either.

# The empirical value 0.080 for L(χ5) is between ζ and L_DH.
# Convergence is slow due to (log T)^4 leading + lower order terms.

# Let me compute a more honest analysis: fit log moment vs log(log T) to extract effective power.
print("Power-law fit: log M_2k vs log(log T) for T in [100, 4000]")
mask = (t_grid > 100) & (t_grid < 4000)
ll = np.log(logT[mask])
# k=1 (2nd moment) — should have slope k^2 = 1
for name, mom in [("ζ M2", I2_z), ("L(χ5) M2", I2_c), ("L_DH M2", I2_d)]:
 lm = np.log(mom[mask])
 slope, intercept = np.polyfit(ll, lm, 1)
 print(f" {name}: slope = {slope:.4f} (theory k^2=1 for unitary)")
print("k=2 (4th moment) — should have slope k^2 = 4")
for name, mom in [("ζ M4", M4_z), ("L(χ5) M4", M4_c), ("L_DH M4", M4_d)]:
 lm = np.log(mom[mask])
 slope, intercept = np.polyfit(ll, lm, 1)
 print(f" {name}: slope = {slope:.4f} (theory k^2=4 for unitary)")


Power-law fit: log M_2k vs log(log T) for T in [100, 4000]
 ζ M2: slope = 1.3328 (theory k^2=1 for unitary)
 L(χ5) M2: slope = 0.9001 (theory k^2=1 for unitary)
 L_DH M2: slope = 0.8033 (theory k^2=1 for unitary)
k=2 (4th moment) — should have slope k^2 = 4
 ζ M4: slope = 4.0553 (theory k^2=4 for unitary)
 L(χ5) M4: slope = 2.8705 (theory k^2=4 for unitary)
 L_DH M4: slope = 2.5335 (theory k^2=4 for unitary)


In [25]:

# Interesting! The slopes show:
# - ζ M4: slope 4.055 — excellent match to RMT power 4
# - ζ M2: slope 1.33 — affected by low-T lower-order terms (M_2 has additive constant)
# - L(χ5) M2: 0.900 / M4: 2.87 — DEVIATES from RMT power
# - L_DH M2: 0.80 / M4: 2.53 — also deviates
# 
# Wait, L(χ5) deviates too? That's because L(χ5) has GRH expected but the moment 
# power should still be k^2. Let me check if I have correct chi.
# 
# Actually for primitive Dirichlet χ mod q, M_{2k}(T) ~ a_k(χ) * g_k * (log T)^{k^2}
# So power IS k^2.
# 
# The deviation for L(χ5) (slope 2.87 vs 4) and L_DH (2.53) is striking.
# Let me check if my L(χ5) implementation is correct. We have zeros validated against LMFDB.
# But the function evaluation - did I get the right character?
# χ_4 mod 5: order-4 character. Values χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1. Let me double-check.
# In LMFDB: χ_5,2 (Conrey label 5.2) is the primitive character of order 4 mod 5.
# χ(2) = i means a generator: 2 is a primitive root mod 5 (2^1=2, 2^2=4, 2^3=3, 2^4=1).
# χ(2^k) = i^k. So χ(2)=i, χ(4)=i^2=-1, χ(3)=i^3=-i, χ(1)=1. Matches.

# Let me verify L(χ5) values match LMFDB:
import mpmath as mp
mp.mp.dps = 25
chi5_vals = [0, 1, 1j, -1j, -1]
# Value of L(χ,1) for this character:
val_at_1 = mp.dirichlet(1, chi5_vals)
print("L(χ,1) =", val_at_1)
# LMFDB says L(χ_5.2, 1) ≈ ?
# Also check completed L-function symmetric — should fit GRH zeros.

# Check first zero of L(χ5) imaginary part matches: 6.6485 (from our list)
val_at_zero1 = mp.dirichlet(mp.mpc(0.5, 6.6484533447), chi5_vals)
print("L(χ,0.5+6.6485i) =", val_at_zero1, "|.|=", abs(val_at_zero1))


L(χ,1) = (0.8648062659772099672311821 + 0.2041530661383851461940023j)
L(χ,0.5+6.6485i) = (0.3465784248523228702732706 + 0.36318452595434823940068j) |.|= 0.502015542056022266431817


In [26]:

# Hmm — L(χ,0.5+6.6485i) should be ≈ 0 at the zero, but I'm getting |.|=0.5. 
# Let me try more digits:
zero1 = mp.mpc(0.5, lchi5[0]) # full precision string
val = mp.dirichlet(zero1, chi5_vals)
print("L(χ,0.5+γ_1 i) full prec =", val, "|.|=", abs(val))
# If this is also nonzero, then maybe character ordering is wrong.


L(χ,0.5+γ_1 i) full prec = (0.3465784248783005813588375 + 0.3631845259647931888720159j) |.|= 0.50201554208151302745353


In [27]:

# So L(χ5, 0.5+6.648i) ≠ 0! My character or function is wrong.
# Let me try the conjugate character (since L(χ,s) and L(χ̄,s) have same zeros — actually 
# different but related; for non-real χ they're different but share γ if we count both):
chi5_conj = [0, 1, -1j, 1j, -1]
val = mp.dirichlet(mp.mpc(0.5, lchi5[0]), chi5_conj)
print("Conjugate L(χ̄,0.5+γ_1 i):", val, "|.|=", abs(val))

# Try other characters mod 5 to find the right one matching our zeros
# Generator g=2 mod 5. The 4 characters: χ_k(2) = i^k for k=0,1,2,3.
# χ_0 trivial, χ_1: 2→i, χ_2: 2→-1 (Legendre), χ_3: 2→-i
# The "χ_5,2" in LMFDB is the one with χ(2)=i.
# 
# Maybe the issue is that mpmath.dirichlet uses chi indexed from 0 with chi[n%q]?
# Let me check mpmath docs.
help(mp.dirichlet)


Conjugate L(χ̄,0.5+γ_1 i): (3.101665528573101113650767 + 0.8092422169119878629478734j) |.|= 3.205495596124144347761697
Help on method dirichlet in module mpmath.functions.zeta:

dirichlet(s, chi=[1], derivative=0) method of mpmath.ctx_mp.MPContext instance
 Evaluates the Dirichlet L-function

 .. math ::

 L(s,\chi) = \sum_{k=1}^\infty \frac{\chi(k)}{k^s}.

 where `\chi` is a periodic sequence of length `q` which should be supplied
 in the form of a list `[\chi(0), \chi(1), \ldots, \chi(q-1)]`.
 Strictly, `\chi` should be a Dirichlet character, but any periodic
 sequence will work.

 For example, ``dirichlet(s, [1])`` gives the ordinary
 Riemann zeta function and ``dirichlet(s, [-1,1])`` gives
 the alternating zeta function (Dirichlet eta function).

 Also the derivative with respect to `s` (currently only a first
 derivative) can be evaluated.

 **Examples**

 The ordinary Riemann zeta function::

 >>> from mpmath import *
 >>> mp.dps = 25; mp.pretty = True
 >>> dirichlet(3, [1]); zet

In [28]:

# So mpmath.dirichlet(s, [chi(0), chi(1), ..., chi(q-1)]) is correct format.
# I had chi5_vals = [0, 1, 1j, -1j, -1] i.e. χ(0)=0, χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1.
# Let's check: χ(2)*χ(3) = i*(-i) = 1 = χ(6 mod 5) = χ(1) = 1. ✓
# χ(2)^2 = -1 = χ(4). ✓
# This IS the order-4 character mod 5.
# 
# But L(χ, 0.5+γ_1 i) = 0.502 ≠ 0. Why?
# 
# Wait, our zero list says γ_1 = 6.6485. But maybe these are NOT the L(χ5, s) zeros but 
# something else? Or maybe these are zeros of the Dedekind zeta function of Q(i)... 
# Or maybe Λ(s, χ) zeros which include also archimedean factor poles.
# 
# Actually, the lchi5 list was validated against LMFDB. Let me check LMFDB directly for 
# Conrey char 5.2 first zero:
print("Our first zero:", lchi5[0])
# The LMFDB lists the first positive zero ordinate of L(χ_{5,2}, s) as 6.6484533447... 
# Yes that's what we have. So our zero list is correct.
# 
# Then why does mpmath.dirichlet(0.5+6.648i, [0,1,i,-i,-1]) ≠ 0?

# Let me test the symmetric pair: L(χ,s) and L(χ̄,s) share zeros only on the line.
# Wait, L(χ̄, s) = conj(L(χ, conj(s))). So for non-real χ, on the critical line s=1/2+it:
# L(χ̄, 1/2+it) = conj(L(χ, 1/2-it))
# These two functions have related zeros but they're different.
# 
# But LMFDB's zero list for L(χ_5.2, s) — those are the actual zeros of L(χ_5.2, s).
# So mpmath.dirichlet should also give zero there. Unless my character labels are off.

# Try χ with χ(2)=-1 (Legendre symbol mod 5):
chi_legendre = [0, 1, -1, -1, 1]
val = mp.dirichlet(mp.mpc(0.5, lchi5[0]), chi_legendre)
print("Legendre L(χ,0.5+γ_1):", val, "|.|=", abs(val))

# Try χ(2)=-i:
chi_minus_i = [0, 1, -1j, 1j, -1]
val = mp.dirichlet(mp.mpc(0.5, lchi5[0]), chi_minus_i)
print("χ(2)=-i L(χ,0.5+γ_1):", val, "|.|=", abs(val))


Our first zero: 6.6484533447277147161232784599793178472985854232444983723747129467
Legendre L(χ,0.5+γ_1): (-1.092670273992454916774344e-26 - 4.220494884199926424990658e-26j) |.|= 4.359645099686956957038766e-26
χ(2)=-i L(χ,0.5+γ_1): (3.101665528573101113650767 + 0.8092422169119878629478734j) |.|= 3.205495596124144347761697


In [29]:

# Aha! The character is actually the LEGENDRE symbol mod 5 (real quadratic character), not the 
# order-4 character. χ(1)=1, χ(2)=-1, χ(3)=-1, χ(4)=1.
# This is "χ_4 mod 5" in the dataset's notation — meaning the character with conductor 4? Or just 
# the unique non-trivial real character mod 5 (which is the Kronecker symbol).
# Specifically, this is χ_5,4 in LMFDB Conrey labelling (Legendre symbol).
# 
# OK so the dataset's "L(χ₄ mod 5)" refers to χ = quadratic char mod 5 (the Kronecker symbol).
# That's a REAL character, so L(χ, s) is a degree-1 L-function attached to ℚ(√5)... no, attached to
# the real quadratic field/extension. It IS a Dirichlet L-function with U(N) symmetry expected.
#
# Now I need to RECOMPUTE |L(χ5,1/2+it)| with the correct character. This will take ~10 min.
# But we're running low on time (~890s left). Let me redo it more efficiently.

# Rewrite helper
helper_code = '''
import mpmath as mp
import numpy as np

mp.mp.dps = 18
chi5_vals = [0, 1, -1, -1, 1] # Kronecker symbol mod 5
kappa = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2) / (mp.sqrt(5) - 1)

def abs_lchi5_at(t):
 return abs(complex(mp.dirichlet(mp.mpc(0.5, t), chi5_vals)))

def batch_lchi5(ts):
 return [abs_lchi5_at(t) for t in ts]
'''
with open('lf_helpers2.py', 'w') as f:
 f.write(helper_code)

# Real-valued character means dirichlet is faster (since it's a real character L can be 
# computed efficiently as a real function via Hurwitz zeta sum)
# Let me also use a coarser dt — dt=0.4, N_pts=10000.
import importlib
if 'lf_helpers2' in dir():
 import lf_helpers2
 importlib.reload(lf_helpers2)
else:
 import lf_helpers2

import numpy as np
t_grid_c = np.arange(0.4, 4000+0.1, 0.4)
print("N_pts:", len(t_grid_c))


N_pts: 10000


In [30]:

# Use parallel processing with the fixed character
from multiprocessing import Pool
import time

n_workers = 8
chunks = [list(t_grid_c[i::n_workers]) for i in range(n_workers)]

t0 = time.time()
with Pool(n_workers) as pool:
 results = pool.map(lf_helpers2.batch_lchi5, chunks)
elapsed = time.time() - t0
print(f"lchi5 (correct char) done in {elapsed:.1f}s")

abs_lchi5 = np.empty(len(t_grid_c))
for i, res in enumerate(results):
 abs_lchi5[i::n_workers] = res
np.save('abs_lchi5_grid_v2.npy', abs_lchi5)
print(f"mean |L(χ5)| = {abs_lchi5.mean():.3f}")


TimeoutError: Code execution timed out after 268.0 seconds